# nnU-Net BraTS 2024 MEN-RT — Full Pipeline

**Before running this notebook:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset (the two zip files) as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `GITHUB_TOKEN`
   - Notebook → Add-ons → Secrets → Add New Secret

**Training plan (pilot — 50 cases subset):**
- Fold 0: 50 epochs → then run inference (fold 0 only, not ensemble) → prompts for foundation model
- Fold 1: 50 epochs (Cell 17, same session or new session)
- Early stopping is active but only kicks in after the 50-epoch warmup

**Checkpoints saved automatically per fold:**
- `checkpoint_best.pth` — best validation Dice
- `checkpoint_latest.pth` — most recent epoch (resume after interruption)
- `checkpoint_final.pth` — written at end of training

**Expected runtime:** ~1–2 hours on T4 GPU for 50 epochs / 50 cases

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Check available input data ───────────────────────────────────────
# Run this to find the correct path to your BraTS zip files
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets  = UserSecretsClient()
token    = secrets.get_secret('nnunet_kaggle')
repo_url = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'

# Remove old clone if exists, then fresh clone
os.system('rm -rf /kaggle/working/nnunet-training')
os.system(f'git clone {repo_url} /kaggle/working/nnunet-training')

print('
Repository contents:')
os.system('ls /kaggle/working/nnunet-training')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
# surface-distance must be installed from GitHub (PyPI version is a different package)
!pip install git+https://github.com/google-deepmind/surface-distance.git -q
!pip install nnunetv2 -q
!pip install nibabel SimpleITK scipy scikit-image pandas matplotlib loguru python-dotenv tqdm rich mlflow -q

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Configure environment variables ───────────────────────────────────
#
# !! EDIT THE TWO PATHS BELOW !!
# Check Cell 2 output to find the correct paths to your zip files.
# Example: /kaggle/input/brats-men-rt/BraTS2024-MEN-RT-TrainingData.zip
#
TRAIN_ZIP = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-TrainingData.zip'
VAL_ZIP   = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-ValidationData.zip'

# ── Subset size (set to None to use ALL cases) ────────────────────────────────
MAX_CASES = 50    # train on only the first 50 cases for this pilot run

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT = '/kaggle/working/nnunet-training'

os.environ['nnUNet_raw']          = f'{PROJECT}/nnunet_raw'
os.environ['nnUNet_preprocessed'] = f'{PROJECT}/nnunet_preprocessed'
os.environ['nnUNet_results']      = f'{PROJECT}/checkpoints'
os.environ['DATASET_ID']          = '001'
os.environ['DATASET_NAME']        = 'BraTSMENRT'
os.environ['NNUNET_CONFIGURATION']= '3d_fullres'
os.environ['NNUNET_SEED']         = '42'
os.environ['ES_PATIENCE']         = '50'
os.environ['ES_MIN_DELTA']        = '0.0001'
os.environ['ES_WARMUP']           = '50'
os.environ['NUM_FOLDS']           = '5'
os.environ['CUDA_VISIBLE_DEVICES']= '0'
os.environ['TRAIN_SOURCE']        = TRAIN_ZIP
os.environ['VAL_SOURCE']          = VAL_ZIP
os.environ['LABEL_SUFFIX']        = 'gtv'
os.environ['EXPERIMENT_NAME']     = 'BraTSMENRT_baseline'
os.environ['MLFLOW_TRACKING_URI'] = f'{PROJECT}/experiments/mlruns'

# Create output directories
for d in ['nnunet_raw','nnunet_preprocessed','checkpoints',
          'metrics','results','visualizations','inference_outputs','logs','experiments']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

print('Environment configured:')
print(f'  nnUNet_raw          : {os.environ["nnUNet_raw"]}')
print(f'  nnUNet_preprocessed : {os.environ["nnUNet_preprocessed"]}')
print(f'  nnUNet_results      : {os.environ["nnUNet_results"]}')
print(f'  TRAIN_SOURCE        : {TRAIN_ZIP}')
print(f'  VAL_SOURCE          : {VAL_ZIP}')
print(f'  MAX_CASES           : {MAX_CASES}')

In [ ]:
# ── Cell 6: Change working directory to the repo ──────────────────────────────
# All ! commands after this run inside the repository folder
%cd /kaggle/working/nnunet-training

In [ ]:
# ── Cell 7: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────────
# Uses only the first MAX_CASES training cases (pilot run = 50 cases).
# Set MAX_CASES = None in Cell 5 to use the full dataset.
# Expected time: 2–5 minutes for 50 cases
!python scripts/01_prepare_dataset.py \
    --train    "{TRAIN_ZIP}" \
    --val      "{VAL_ZIP}"   \
    --max-cases {MAX_CASES}  \
    --log-dir  logs

In [ ]:
# Verify dataset was created correctly
!ls -lh $nnUNet_raw/Dataset001_BraTSMENRT/
!echo '--- imagesTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/imagesTr/ | wc -l
!echo '--- labelsTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/labelsTr/ | wc -l
!echo '--- dataset.json ---'
!cat $nnUNet_raw/Dataset001_BraTSMENRT/dataset.json

In [ ]:
# ── Cell 8: STEP 2 — nnU-Net Planning and Preprocessing ──────────────────────
# nnU-Net reads the data and automatically decides:
#   - patch size, batch size, network depth
#   - resampling spacing, normalization
# Expected time: 20–40 minutes
!python scripts/02_preprocess.py --log-dir logs

In [ ]:
# Verify preprocessing output
!ls $nnUNet_preprocessed/Dataset001_BraTSMENRT/
!echo '--- Splits file (5-fold CV) ---'
!python -c "import json; s=json.load(open('$nnUNet_preprocessed/Dataset001_BraTSMENRT/splits_final.json')); print(f'{len(s)} folds | fold_0 train={len(s[0][\"train\"])} val={len(s[0][\"val\"])}')" 2>/dev/null || echo 'splits_final.json not found yet'

In [ ]:
# ── Cell 9: STEP 3 — Train Fold 0 (50 epochs) ───────────────────────────────
#
# Trains Fold 0 on the 50-case subset for 50 epochs.
# nnU-Net automatically saves:
#   checkpoint_best.pth   ← best validation Dice (saved whenever Dice improves)
#   checkpoint_latest.pth ← saved every epoch  (safe to resume if interrupted)
#   checkpoint_final.pth  ← written at the end of training
#
# Early stopping: patience=50, warmup=50, min_delta=0.0001
# (with only 50 epochs total it won't trigger, but it's wired in)
#
# If interrupted, re-run Cells 1-6, then add --c to resume from checkpoint_latest:
#   !nnUNetv2_train 001 3d_fullres 0 -tr nnUNetTrainerEarlyStopping -p nnUNetPlans --num_epochs 50 --c
#
# Expected time: ~1-2 hours on T4 GPU
#
!nnUNetv2_train 001 3d_fullres 0 \n    -tr nnUNetTrainerEarlyStopping \n    -p  nnUNetPlans \n    --num_epochs 50


In [ ]:
# Verify checkpoints were saved
import os
CKPT_DIR = '/kaggle/working/nnunet-training/checkpoints/Dataset001_BraTSMENRT/nnUNetTrainerEarlyStopping__nnUNetPlans__3d_fullres/fold_0'
print(f'Checkpoint directory: {CKPT_DIR}')
!ls -lh "{CKPT_DIR}" 2>/dev/null || echo 'Not found — training may have failed'
!echo ''
!echo 'Expected files:'
!echo '  checkpoint_best.pth   <- best val Dice'
!echo '  checkpoint_latest.pth <- most recent epoch (use --c to resume)'
!echo '  checkpoint_final.pth  <- written at end of training'

# Print latest training log metrics
import glob
log_files = sorted(glob.glob(f'{CKPT_DIR}/training_log*.txt'))
if log_files:
    print(f'
Last 20 lines of {log_files[-1]}:')
    with open(log_files[-1]) as fh:
        lines = fh.readlines()
    print(''.join(lines[-20:]))


In [ ]:
# ── Cell 10: STEP 4 — Inference using Fold 0 only ────────────────────────────
#
# Predicts GTV masks for the held-out validation cases of fold 0,
# using ONLY fold 0's trained model (no ensemble, no other folds).
#
# These predictions will be used as segmentation prompts for the next
# foundation model to refine segmentation.
#
# Output: inference_outputs/fold0_prompts/fold_0/*.nii.gz
# Expected time: 10–30 minutes
!python scripts/04_inference.py \
    --cv-mode \
    --folds 0 \
    --cv-output-root inference_outputs/fold0_prompts \
    --log-dir logs

In [ ]:
# Verify fold 0 prompt predictions exist
!echo '--- Fold 0 prompt predictions (for foundation model) ---'
!ls inference_outputs/fold0_prompts/fold_0/ 2>/dev/null | head -10
!echo ''
!echo 'Total predictions (fold 0 val cases):'
!ls inference_outputs/fold0_prompts/fold_0/*.nii.gz 2>/dev/null | wc -l
!echo ''
!echo '--- Manifest ---'
!cat inference_outputs/fold0_prompts/cv_prediction_manifest.json 2>/dev/null || echo 'manifest not found'

In [ ]:
# ── Cell 11: STEP 5 — Evaluate Fold 0 ────────────────────────────────────────
# Computes: DSC, HD95, NSD, Precision, Recall, Specificity, Volume Error
# Expected time: 5–15 minutes
!python scripts/05_evaluate.py \
    --cv-mode \
    --results-dir results \
    --show-rankings \
    --log-dir logs

In [ ]:
# ── Cell 12: STEP 6 — Visualizations ─────────────────────────────────────────
# Generates: overlays, violin plots, fold comparison charts, training curves
# Expected time: 2–5 minutes
!python scripts/06_visualize.py \
    --all \
    --cv-mode \
    --results-dir results \
    --metrics-dir metrics \
    --output-dir visualizations \
    --log-dir logs

In [ ]:
# ── Cell 13: View Results ─────────────────────────────────────────────────────
import pandas as pd

result_files = [
    'results/fold_0_per_case.csv',
    'results/cv_combined.csv',
]

for f in result_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        cols = ['case_id','dice','hd95','nsd','precision','recall']
        cols = [c for c in cols if c in df.columns]
        print(f'\n=== {f} ({len(df)} cases) ===')
        print(df[cols].round(4).to_string(index=False))
        print(f'\nMean DSC  : {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
        if 'hd95' in df.columns:
            finite_hd = df['hd95'].replace([float('inf')], float('nan'))
            print(f'Mean HD95 : {finite_hd.mean():.2f} mm')
        if 'nsd' in df.columns:
            print(f'Mean NSD  : {df["nsd"].mean():.4f}')
    else:
        print(f'{f} not found yet')

In [ ]:
# ── Cell 14: Show overlay images ──────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

overlay_dir = Path('visualizations/overlays')
if overlay_dir.exists():
    images = sorted(overlay_dir.glob('*.png'))[:3]  # show first 3
    for img_path in images:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))
else:
    print('No overlay images found — run Cell 12 first')

In [ ]:
# ── Cell 15: Show metric plots ────────────────────────────────────────────────
for plot in ['visualizations/metrics_violin.png',
             'visualizations/metrics_boxplot.png',
             'visualizations/volume_scatter.png']:
    if os.path.exists(plot):
        print(plot)
        display(Image(plot, width=900))

In [ ]:
# ── Cell 16: Save everything to Kaggle output ─────────────────────────────────
# Kaggle deletes /kaggle/working/nnunet-training when the session ends.
# This cell copies all important outputs to /kaggle/working/ which IS saved.
import shutil, os

SAVE = '/kaggle/working/outputs'
os.makedirs(SAVE, exist_ok=True)

to_save = [
    ('results',                        f'{SAVE}/results'),
    ('metrics',                        f'{SAVE}/metrics'),
    ('visualizations',                 f'{SAVE}/visualizations'),
    ('inference_outputs',              f'{SAVE}/inference_outputs'),
    # Fold 0 only predictions — prompts for foundation model
    ('inference_outputs/fold0_prompts',f'{SAVE}/fold0_prompts'),
    # Checkpoints: checkpoint_best.pth, checkpoint_latest.pth, checkpoint_final.pth
    ('checkpoints',                    f'{SAVE}/checkpoints'),
]

for src, dst in to_save:
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {src} -> {dst}')
    else:
        print(f'Skipped (not found): {src}')

print('
All outputs saved to /kaggle/working/outputs/')
!du -sh /kaggle/working/outputs/*

# Show checkpoint files per fold
import glob
ckpt_root = f'{SAVE}/checkpoints'
for ckpt in sorted(glob.glob(f'{ckpt_root}/**/*.pth', recursive=True)):
    print(f'  Checkpoint: {ckpt}')

---

## After Fold 0: Inference → Foundation Model Prompts

After Cell 9 (train fold 0) completes:
1. Run **Cell 10** → inference with fold 0 model only → `inference_outputs/fold0_prompts/fold_0/*.nii.gz`
2. These `.nii.gz` files are your **segmentation prompts** for the next foundation model
3. Run Cells 11–16 to evaluate metrics and save outputs

## Train Fold 1 (Cell 17 below)

Run Cell 17 **after fold 0 + inference are done**.
If in a **new Kaggle session**: re-run Cells 1–6 (skip 7–8 if already preprocessed), then run Cell 17.

> **Checkpoint resume tip:** If training is interrupted, add `--c` flag:
> ```
> nnUNetv2_train 001 3d_fullres 0 -tr nnUNetTrainerEarlyStopping --num_epochs 50 --c
> ```
> nnU-Net will resume from `checkpoint_latest.pth` automatically.

In [ ]:
# ── Cell 17: Train Fold 1 (50 epochs) ────────────────────────────────────────
#
# Run this cell AFTER fold 0 training + inference are done.
# Trains fold 1 for 50 epochs with the same settings as fold 0.
# Checkpoints are saved to: checkpoints/Dataset001_BraTSMENRT/.../fold_1/
#   checkpoint_best.pth   ← best val Dice
#   checkpoint_latest.pth ← most recent epoch
#   checkpoint_final.pth  ← saved at end of training
#
# If running in a new session: re-run Cells 1–10 first (skip Cell 7/8 if
# already done), then run this cell.
#
import subprocess, sys

FOLD = 1
print(f'Starting Fold {FOLD} — 50 epochs')
cmd = [
    sys.executable, '-m', 'nnunetv2.run.run_training',
    '001', '3d_fullres', str(FOLD),
    '-tr', 'nnUNetTrainerEarlyStopping',
    '-p',  'nnUNetPlans',
    '--num_epochs', '50',
]
rc = subprocess.run(cmd).returncode
print(f'Fold {FOLD} finished with return code {rc}')